# Notebook 04 — Rigorous 3-Model Evaluation & Diagnostics

Empirical comparison across three systems:
1. **Base LAYA** (`multilingual` checkpoint)
2. **MedNLI Fine-Tuned LAYA** (`pavanmantha/mednli_laya`)
3. **CEIS Multi-Task Model** (`runs/ceis_seed0`)

### Evaluated Metrics:
- Accuracy & 95% Bootstrap Confidence Intervals
- Macro F1 & Confusion Matrices
- Brier Score & Expected Calibration Error (ECE)
- Order-Flip Robustness (reversing option definitions)
- Claim-Only Artefact Control

In [ ]:
import sys, json
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.insert(0, str(Path('..').resolve()))
from evaluation.benchmark import Benchmark, load_eval_config
from evaluation.metrics import LABELS

## 1 · Execute Full Benchmark

In [ ]:
bench = Benchmark()
results = bench.run()
print('Benchmark evaluation completed.')

## 2 · Performance Comparison Table

In [ ]:
rows = []
for m in results['models']:
    ci = m['accuracy_ci95']
    rows.append({
        'Model': m['name'],
        'Accuracy': f"{m['accuracy']:.4f}",
        '95% CI': f"[{ci[0]:.4f}, {ci[1]:.4f}]",
        'Macro F1': f"{m['macro_f1']:.4f}",
        'Brier Score': f"{m['brier']:.4f}",
        'ECE': f"{m['ece']:.4f}",
        'Order Flip Rate': f"{m['order_flip_rate']:.4f}",
        'Claim-Only Acc': f"{m['claim_only_acc']:.4f}",
        'Latency (ms)': f"{m['latency_ms']:.1f}",
    })
summary_df = pd.DataFrame(rows)
display(summary_df)

## 3 · Confusion Matrix Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for idx, m in enumerate(results['models']):
    cm = np.array(m['confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=LABELS, yticklabels=LABELS, ax=axes[idx])
    axes[idx].set_title(m['name'])
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('True Label')
plt.tight_layout()
plt.show()